# Wanderbricks — Explore Bronze & Silver

Interactive exploration of the raw (bronze) and cleaned (silver) GSOD tables.

**Prereqs:** attach a compute (serverless), run the DLT pipeline once so the tables exist, then run cells top to bottom. Use the widgets at the top to switch catalog/schema (dev = workspace / your username).

In [ ]:
# --- Bootstrap: run in the workspace OR locally (Databricks Connect + SDK) ---
# In the workspace: spark/dbutils exist natively.
# Locally: Databricks Connect (serverless) + databricks-sdk emulate them.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.profile("irajput").serverless().getOrCreate()

try:
    dbutils
except NameError:
    from databricks.sdk import WorkspaceClient
    _w = WorkspaceClient(profile="irajput")
    dbutils = _w.dbutils

try:
    dbutils.widgets.text("catalog", "workspace")
    dbutils.widgets.text("schema", "iraonfridays")
    CATALOG = dbutils.widgets.get("catalog").strip()
    SCHEMA = dbutils.widgets.get("schema").strip()
except Exception:
    CATALOG = "workspace"
    SCHEMA = "iraonfridays"

print(f"Connected to Databricks! Target: {CATALOG}.{SCHEMA}")

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col

def table_exists(name: str) -> bool:
    return spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{name}")

for t in ["gsod_bronze", "gsod_silver", "weather_features", "forecasts"]:
    print(f"{t:20s} {'EXISTS' if table_exists(t) else 'MISSING - run the DLT pipeline first'}")

## Sample rows from silver

In [ ]:
if table_exists("gsod_silver"):
    display(spark.table(f"{CATALOG}.{SCHEMA}.gsod_silver").limit(20))
else:
    print("gsod_silver missing - run the DLT pipeline first")

## Overview: rows, stations, date range

In [ ]:
if table_exists("gsod_silver"):
    display(
        spark.table(f"{CATALOG}.{SCHEMA}.gsod_silver")
        .agg(
            F.count("*").alias("rows"),
            F.countDistinct("station").alias("stations"),
            F.min("date").alias("min_date"),
            F.max("date").alias("max_date"),
        )
    )

## Missing values (post-cleanup)

In [ ]:
if table_exists("gsod_silver"):
    silver = spark.table(f"{CATALOG}.{SCHEMA}.gsod_silver")
    null_counts = [
        F.count(F.when(col(c).isNull(), 1)).alias(f"{c}_null")
        for c in ["temp_c", "max_c", "min_c", "dewp_c", "prcp_mm", "wdsp_knots"]
    ]
    display(silver.agg(F.count("*").alias("rows"), *null_counts))

## Top stations by record count

In [ ]:
if table_exists("gsod_silver"):
    display(
        spark.table(f"{CATALOG}.{SCHEMA}.gsod_silver")
        .groupBy("station", "station_name")
        .count()
        .orderBy(F.desc("count"))
        .limit(10)
    )

## One station's temperature series

In [ ]:
dbutils.widgets.text("station", "", "station (empty = top by count)")

if table_exists("gsod_silver"):
    station = dbutils.widgets.get("station").strip()
    if not station:
        station = (
            spark.table(f"{CATALOG}.{SCHEMA}.gsod_silver")
            .groupBy("station").count().orderBy(F.desc("count"))
            .limit(1).collect()[0][0]
        )
    print(f"station: {station}")
    display(
        spark.table(f"{CATALOG}.{SCHEMA}.gsod_silver")
        .filter(F.col("station") == station)
        .select("date", "temp_c", "max_c", "min_c")
        .orderBy("date")
    )

## Try it

- Change the `station` widget and re-run the last cell.
- Compare bronze vs silver row counts: bronze keeps every raw row; silver drops rows that fail the `@dlt.expect_or_drop` gates.
- Values are now real units (°C, mm, knots); raw GSOD stores tenths.